In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & DATABASE CONNECTION
# =============================================================================
import sys
import os
import importlib
import pandas as pd
import numpy as np
import time
import datetime
from sqlalchemy import create_engine, text

# Add the parent directory to sys.path to allow importing from main directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Import and reload database initialization module
import init_database
importlib.reload(init_database)

# Establish connection
engine = init_database.init_db()

print("✅ Environment ready and database connection established!")

In [ ]:
def load_gold(engine):
    try:
        batch_start_time = time.time()
        print('================================================')
        print('Loading gold Layer')
        print('================================================')

        # --- 1. Create Dimension: gold.dim_customers---
        print('------------------------------------------------')
        print('Loading dim_customers Table')
        print('------------------------------------------------')      
        
        start_time = time.time()

        df_cust_info = pd.read_sql_query("SELECT * FROM silver.crm_cust_info", engine)
        df_cust_az12 = pd.read_sql_query("SELECT * FROM silver.erp_cust_az12", engine)
        df_loc_a101 = pd.read_sql_query("SELECT * FROM silver.erp_loc_a101", engine)

        # Join CRM info with ERP Customer & Location data
        df_dim_customers = pd.merge(df_cust_info, df_cust_az12, left_on='cst_key', right_on='CID', how='left')
        df_dim_customers = pd.merge(df_dim_customers, df_loc_a101, left_on='cst_key', right_on='CID', how='left')

        # Apply Business Logic: Gender Fallback (CRM -> ERP
        df_dim_customers['gender'] = np.where(
            df_dim_customers['cst_gndr'] != 'n/a',
            df_dim_customers['cst_gndr'],
            df_dim_customers['GEN'].fillna('n/a')
        )

        # Selecting & Renaming Columns to business-friendly names
        df_dim_customers = df_dim_customers[[
            'cst_id', 'cst_key', 'cst_firstname', 'cst_lastname', 
            'CNTRY', 'cst_marital_status', 'gender', 'BDATE', 'cst_create_date'
        ]].rename(columns={
            'cst_id': 'customer_id',
            'cst_key': 'customer_number',
            'cst_firstname': 'first_name',
            'cst_lastname': 'last_name',
            'CNTRY': 'country',
            'cst_marital_status': 'marital_status',
            'BDATE': 'birthdate',
            'cst_create_date': 'create_date'
        })

        # Create Surrogate Key (customer_key)
        df_dim_customers.insert(0, 'customer_sk', range(1, len(df_dim_customers) + 1))

        # Load to Gold Layer
        df_dim_customers.to_sql('dim_customers', engine, schema='gold', if_exists='replace', index=False)

        row_count = len(df_dim_customers)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        # --- 1. Create Dimension: gold.dim_products---
        print('------------------------------------------------')
        print('Loading dim_products Table')
        print('------------------------------------------------')      
        
        start_time = time.time()

        df_prd_info = pd.read_sql("SELECT * FROM silver.crm_prd_info", engine)
        df_erp_px = pd.read_sql("SELECT * FROM silver.erp_px_cat_g1v2", engine) 

        # Join CRM Products with ERP Categories
        df_dim_products = pd.merge(df_prd_info, df_erp_px, left_on='cat_id', right_on='ID', how='left')

        # Filter for active products only (Latest records)
        df_dim_products = df_dim_products.loc[df_dim_products['prd_end_dt'].isnull()].copy()

        # Selecting & Renaming Columns
        df_dim_products = df_dim_products[[
            'prd_id', 'prd_key', 'prd_nm', 'cat_id', 'CAT', 'SUBCAT', 
            'MAINTENANCE', 'prd_cost', 'prd_line', 'prd_start_dt'
        ]].rename(columns={
            'prd_id': 'product_id',
            'prd_key': 'product_number',
            'prd_nm': 'product_name',
            'cat_id': 'category_id',
            'CAT': 'category',
            'SUBCAT': 'subcategory',
            'MAINTENANCE': 'maintenance',
            'prd_cost': 'cost',
            'prd_line': 'product_line',
            'prd_start_dt': 'start_date'
        })

        # Create Surrogate Key (product_key)
        df_dim_products.insert(0, 'product_sk', range(1, len(df_dim_products) + 1))

        # Load to Gold Layer
        df_dim_products.to_sql('dim_products', engine, schema='gold', if_exists='replace', index=False)

        row_count = len(df_dim_products)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        # --- 1. Create Dimension: gold.fact_sales---
        print('------------------------------------------------')
        print('Loading Fact_Sales Table')
        print('------------------------------------------------') 

        start_time = time.time()

        df_sales = pd.read_sql("SELECT * FROM silver.crm_sales_details", engine)
        df_dim_products = pd.read_sql("SELECT * FROM gold.dim_products", engine)
        df_dim_customers = pd.read_sql("SELECT * FROM gold.dim_customers", engine)

        # Link Sales with Products Dimension to get product_key
        df_fact_sales = df_sales.merge(df_dim_products[['product_sk', 'product_number']],
                                    left_on='sls_prd_key', right_on='product_number', how='left')
        
        # Link Sales with Customers Dimension to get customer_key
        df_fact_sales = pd.merge(df_fact_sales, df_dim_customers[['customer_sk', 'customer_id']], 
                                 left_on='sls_cust_id', right_on='customer_id', how='left')
        
        # Selecting & Renaming Columns
        df_fact_sales = df_fact_sales[[
            'sls_ord_num', 'product_sk', 'customer_sk', 'sls_order_dt', 
            'sls_ship_dt', 'sls_due_dt', 'sls_sales', 'sls_quantity', 'sls_price'
        ]].rename(columns={
            'sls_ord_num': 'order_number',
            'sls_order_dt': 'order_date',
            'sls_ship_dt': 'shipping_date',
            'sls_due_dt': 'due_date',
            'sls_sales': 'sales_amount',
            'sls_quantity': 'quantity',
            'sls_price': 'price'
        })

        df_fact_sales.to_sql('fact_sales', engine, schema='gold', if_exists='replace', index=False)

        row_count = len(df_fact_sales)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        print('==========================================')
        print(f'Loading Silver Layer Completed in {round(time.time()-batch_start_time, 2)} sec')
        print('==========================================')



    except Exception as e:
        print('==========================================')
        print('ERROR OCCURED DURING LOADING GOLD LAYER')
        print(f'Error Message: {str(e)}')
        print('==========================================')

# Execute the process
if __name__ == "__main__":

    # If the engine is successfully created, proceed to load the gold layer
    if engine:
        load_gold(engine)
    else:
        print("❌ Failed to connect to database. Aborting load.")
